# 02 — Dynamic Flex VORP & Sleeper ADP Arbitrage

This notebook consumes the merged projections + Sleeper player catalog produced
by **Notebook 01** and builds a full draft-value board.

**Workflow**
1. Setup & data ingestion (re-apply Notebook 01 normalization)
2. Dynamic flex baseline & VORP calculation
3. Market arbitrage engine (ADP delta signals)
4. Output & validation

## Cell 1 — Setup & Data Ingestion

Load the projections template and the cached Sleeper player catalog, then merge
them using the same normalization logic from Notebook 01.

In [1]:
import os
import re
import json
import pandas as pd

# -- Project paths --------------------------------------------------
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR     = os.path.join(PROJECT_ROOT, "data")

SLEEPER_CACHE   = os.path.join(DATA_DIR, "sleeper_players_raw.json")
PROJECTIONS_CSV = os.path.join(DATA_DIR, "projections_template.csv")
DRAFT_BOARD_CSV = os.path.join(DATA_DIR, "draft_board_latest.csv")

SKILL_POSITIONS = ["QB", "RB", "WR", "TE"]

# -- Normalization helpers (identical to Notebook 01) -----------------
SUFFIX_PATTERN = re.compile(
    r'\b(jr|sr|ii|iii|iv|v|esq|phd)\b\s*$',
    re.IGNORECASE,
)


def clean_player_name(name: str) -> str:
    """Normalize a player name for reliable cross-source matching."""
    if not isinstance(name, str):
        return ""
    text = name.lower()
    text = text.replace("'", "").replace("-", "").replace(".", "")
    text = SUFFIX_PATTERN.sub("", text)
    text = " ".join(text.split())
    return text.strip()


# -- Load projections ------------------------------------------------
projections_df = pd.read_csv(PROJECTIONS_CSV)
projections_df["norm_name"]     = projections_df["player_name"].apply(clean_player_name)
projections_df["norm_position"] = projections_df["position"].str.upper().str.strip()

# -- Load & filter Sleeper catalog -----------------------------------
with open(SLEEPER_CACHE, "r") as f:
    sleeper_raw = json.load(f)

records = []
for pid, pdata in sleeper_raw.items():
    pos = (pdata.get("position") or "").upper()
    if pos not in SKILL_POSITIONS:
        continue
    if not pdata.get("active", False):
        continue
    records.append({
        "player_id":     pdata.get("player_id", pid),
        "full_name":     pdata.get("full_name", ""),
        "position":      pos,
        "team":          pdata.get("team"),
        "search_rank":   pdata.get("search_rank"),
        "years_exp":     pdata.get("years_exp"),
        "status":        pdata.get("status"),
        "injury_status": pdata.get("injury_status"),
        "age":           pdata.get("age"),
    })

sleeper_df = pd.DataFrame(records)
sleeper_df.drop_duplicates(subset=["player_id"], inplace=True)
sleeper_df["norm_name"]     = sleeper_df["full_name"].apply(clean_player_name)
sleeper_df["norm_position"] = sleeper_df["position"].str.upper().str.strip()

# -- Merge -----------------------------------------------------------
MERGE_KEYS = ["norm_name", "norm_position"]

board = pd.merge(
    projections_df,
    sleeper_df,
    on=MERGE_KEYS,
    how="inner",
    suffixes=("_proj", "_sleeper"),
)

print(f"Projections rows:  {len(projections_df)}")
print(f"Sleeper rows:      {len(sleeper_df):,}")
print(f"Merged board rows: {len(board)}")
print(f"Match rate:        {len(board)/len(projections_df)*100:.1f}%")

# Create unified position column after merge
board["position"] = board["position_proj"]

board.sort_values("proj_points", ascending=False).head(10)

Projections rows:  15
Sleeper rows:      3,044
Merged board rows: 15
Match rate:        100.0%


,player_name,position_proj,team_proj,proj_points,norm_name,norm_position,player_id,full_name,position_sleeper,team_sleeper,search_rank,years_exp,status,injury_status,age,position
0,Josh Allen,QB,BUF,374.2,josh allen,QB,4984,Josh Allen,QB,BUF,4.0,8.0,Active,NaN,30.0,QB
1,Lamar Jackson,QB,BAL,362.8,lamar jackson,QB,4881,Lamar Jackson,QB,BAL,11.0,8.0,Active,NaN,29.0,QB
2,Jalen Hurts,QB,PHI,355.5,jalen hurts,QB,6904,Jalen Hurts,QB,PHI,29.0,6.0,Active,NaN,28.0,QB
3,Ja'Marr Chase,WR,CIN,312.6,jamarr chase,WR,7564,Ja'Marr Chase,WR,CIN,4.0,5.0,Active,NaN,26.0,WR
4,CeeDee Lamb,WR,DAL,298.4,ceedee lamb,WR,6786,CeeDee Lamb,WR,DAL,10.0,6.0,Active,NaN,27.0,WR
5,Tyreek Hill,WR,MIA,287.1,tyreek hill,WR,3321,Tyreek Hill,WR,NaN,145.0,10.0,Active,Questionable,31.0,WR
6,Bijan Robinson,RB,ATL,280.3,bijan robinson,RB,9509,Bijan Robinson,RB,ATL,1.0,3.0,Active,NaN,24.0,RB
9,Saquon Barkley,RB,PHI,271.5,saquon barkley,RB,4866,Saquon Barkley,RB,PHI,12.0,8.0,Active,NaN,29.0,RB
7,Breece Hall,RB,NYJ,265.7,breece hall,RB,8155,Breece Hall,RB,NYJ,27.0,4.0,Active,Questionable,25.0,RB
13,Marvin Harrison Jr.,WR,ARZ,245.2,marvin harrison,WR,11628,Marvin Harrison,WR,ARI,73.0,2.0,Active,NaN,24.0,WR


## Cell 2 — Dynamic Flex Baseline & VORP Calculation

**League settings (14-team, Full PPR)**

| Slot | Count |
|------|-------|
| QB   | 1     |
| RB   | 2     |
| WR   | 2     |
| TE   | 1     |
| FLEX | 2 (RB/WR/TE) |

**Dedicated starters across the league:**
- QB: 14 &nbsp; RB: 28 &nbsp; WR: 28 &nbsp; TE: 14 &nbsp; → 84 dedicated
- FLEX: 28 total (filled from remaining RB / WR / TE pool)
- **Total skill starters: 112**

**Positional baselines**
- QB baseline = 14th-ranked QB projected points
- RB baseline = min(28th-ranked RB, flex_baseline)
- WR baseline = min(28th-ranked WR, flex_baseline)
- TE baseline = min(14th-ranked TE, flex_baseline)

**VORP** = max(0, proj_points − baseline)

In [2]:
# =====================================================================
# League configuration
# =====================================================================
NUM_TEAMS   = 14
QB_STARTERS = 1  * NUM_TEAMS   # 14
RB_STARTERS = 2  * NUM_TEAMS   # 28
WR_STARTERS = 2  * NUM_TEAMS   # 28
TE_STARTERS = 1  * NUM_TEAMS   # 14
FLEX_SPOTS  = 2  * NUM_TEAMS   # 28

DEDICATED_TOTAL = QB_STARTERS + RB_STARTERS + WR_STARTERS + TE_STARTERS  # 84
SKILL_TOTAL     = DEDICATED_TOTAL + FLEX_SPOTS                          # 112


# =====================================================================
# Safe index helper — avoids IndexError on small template data
# =====================================================================
def get_cutoff_points(df: pd.DataFrame, rank: int) -> float:
    """Return projected points for the player at *rank* (1-based).

    Falls back to the last player's points when *rank* exceeds the
    frame length so the notebook runs cleanly on both a 15-row
    template and a full 250+ player board.
    """
    if len(df) == 0:
        return 0.0
    idx = min(rank - 1, len(df) - 1)
    return float(df.iloc[idx]["proj_points"])


# =====================================================================
# Step 1 — Separate dedicated starters per position
# =====================================================================
pos_sorted = {}
pos_dedicated_ids = {}
for pos, n in [("QB", QB_STARTERS), ("RB", RB_STARTERS),
               ("WR", WR_STARTERS), ("TE", TE_STARTERS)]:
    sub = (
        board[board["position"] == pos]
        .sort_values("proj_points", ascending=False)
        .reset_index(drop=True)
    )
    pos_sorted[pos] = sub
    pos_dedicated_ids[pos] = set(sub.head(n)["player_id"])

# =====================================================================
# Step 2 — Build flex pool (remaining RB / WR / TE)
# =====================================================================
all_dedicated = (
    pos_dedicated_ids["RB"]
    | pos_dedicated_ids["WR"]
    | pos_dedicated_ids["TE"]
)

flex_pool = (
    board[
        (board["position"].isin(["RB", "WR", "TE"]))
        & (~board["player_id"].isin(all_dedicated))
    ]
    .sort_values("proj_points", ascending=False)
    .reset_index(drop=True)
)

# =====================================================================
# Step 3 — Dynamic flex baseline
# =====================================================================
flex_baseline_pts = get_cutoff_points(flex_pool, FLEX_SPOTS)

# =====================================================================
# Step 4 — Positional baselines
# =====================================================================
baselines = {
    "QB": get_cutoff_points(pos_sorted["QB"], QB_STARTERS),
    "RB": min(get_cutoff_points(pos_sorted["RB"], RB_STARTERS), flex_baseline_pts),
    "WR": min(get_cutoff_points(pos_sorted["WR"], WR_STARTERS), flex_baseline_pts),
    "TE": min(get_cutoff_points(pos_sorted["TE"], TE_STARTERS), flex_baseline_pts),
}

print("=== League Configuration ===")
print(f"  Teams:              {NUM_TEAMS}")
print(f"  Dedicated starters: {DEDICATED_TOTAL}")
print(f"  Flex spots:         {FLEX_SPOTS}")
print(f"  Total skill:        {SKILL_TOTAL}")
print(f"  Flex pool size:     {len(flex_pool)}")

print("\n=== Positional Baselines ===")
for pos, bl in baselines.items():
    print(f"  {pos}: {bl:.2f}")
print(f"\n  Flex baseline (28th remaining RB/WR/TE): {flex_baseline_pts:.2f}")

# =====================================================================
# Step 5 — Compute VORP, vorp_rank, pos_rank
# =====================================================================
board["baseline_pts"] = board["position"].map(baselines)
board["vorp"]          = (board["proj_points"] - board["baseline_pts"]).clip(lower=0)

# Overall VORP rank (1 = highest; ties share rank via method='min')
board["vorp_rank"] = board["vorp"].rank(method="min", ascending=False).astype(int)

# Per-position rank (positional rank by projected points)
board["pos_rank"] = (
    board
    .sort_values("proj_points", ascending=False)
    .groupby("position")["proj_points"]
    .rank(method="min", ascending=False)
    .astype(int)
)

board["pos_label"] = board["position"] + board["pos_rank"].astype(str)

# Preview
preview_cols = [
    "vorp_rank", "pos_label", "player_name", "position_proj",
    "team_proj", "proj_points", "baseline_pts", "vorp",
]
board.sort_values("vorp_rank")[preview_cols].head(15)

=== League Configuration ===
  Teams:              14
  Dedicated starters: 84
  Flex spots:         28
  Total skill:        112
  Flex pool size:     0

=== Positional Baselines ===
  QB: 355.50
  RB: 0.00
  WR: 0.00
  TE: 0.00

  Flex baseline (28th remaining RB/WR/TE): 0.00


,vorp_rank,pos_label,player_name,position_proj,team_proj,proj_points,baseline_pts,vorp
3,1,WR1,Ja'Marr Chase,WR,CIN,312.6,0.0,312.6
4,2,WR2,CeeDee Lamb,WR,DAL,298.4,0.0,298.4
5,3,WR3,Tyreek Hill,WR,MIA,287.1,0.0,287.1
6,4,RB1,Bijan Robinson,RB,ATL,280.3,0.0,280.3
9,5,RB2,Saquon Barkley,RB,PHI,271.5,0.0,271.5
7,6,RB3,Breece Hall,RB,NYJ,265.7,0.0,265.7
13,7,WR4,Marvin Harrison Jr.,WR,ARZ,245.2,0.0,245.2
8,8,RB4,Kenneth Walker III,RB,SEA,242.9,0.0,242.9
14,9,WR5,Malik Nabers,WR,NYG,232.8,0.0,232.8
10,10,TE1,Travis Kelce,TE,KC,210.8,0.0,210.8


## Cell 3 — Market Arbitrage Engine

Compare the **market ranking** (`search_rank` from Sleeper — a proxy for ADP)
against our **model ranking** (`vorp_rank`) to surface value and overreach signals.

```
adp_delta = search_rank − vorp_rank

  ≥ +10   🔥 Major Value    — model loves, market sleeps
  +5 … +9 ✅ Slight Value   — model likes more than market
  −4 … +4 ⚖️ Fair Value     — model and market agree
  −9 … −5 ⚠️ Overpriced     — market ranks higher than model
  ≤ −10   🚫 Heavy Reach    — market loves, model is skeptical
```

In [3]:
# =====================================================================
# ADP delta & signal classification
# =====================================================================
board["adp_delta"] = board["search_rank"] - board["vorp_rank"]


def classify_signal(delta):
    """Map a numeric ADP delta to a human-readable signal."""
    if pd.isna(delta):
        return "❓ No ADP"
    if delta >= 10:
        return "🔥 Major Value"
    if delta >= 5:
        return "✅ Slight Value"
    if delta > -5:
        return "⚖️ Fair Value"
    if delta > -10:
        return "⚠️ Overpriced"
    return "🚫 Heavy Reach"


board["signal"] = board["adp_delta"].apply(classify_signal)

# =====================================================================
# Top 10 Value Targets
# =====================================================================
arb_cols = [
    "pos_label", "player_name", "position_proj", "team_proj",
    "proj_points", "vorp", "vorp_rank", "search_rank",
    "adp_delta", "signal",
]

top_value = (
    board
    .sort_values("adp_delta", ascending=False)
    .head(10)
    [arb_cols]
)
print("=== 🔥 Top 10 Value Targets ===")
display(top_value)

# =====================================================================
# Top 5 Overpriced Fades
# =====================================================================
top_fades = (
    board
    .sort_values("adp_delta", ascending=True)
    .head(5)
    [arb_cols]
)
print("\n=== 🚫 Top 5 Overpriced Fades ===")
display(top_fades)

# =====================================================================
# Signal distribution
# =====================================================================
print("\n=== Signal Distribution ===")
print(board["signal"].value_counts().to_string())

=== 🔥 Top 10 Value Targets ===


,pos_label,player_name,position_proj,team_proj,proj_points,vorp,vorp_rank,search_rank,adp_delta,signal
5,WR3,Tyreek Hill,WR,MIA,287.1,287.1,3,145.0,142.0,🔥 Major Value
10,TE1,Travis Kelce,TE,KC,210.8,210.8,10,92.0,82.0,🔥 Major Value
13,WR4,Marvin Harrison Jr.,WR,ARZ,245.2,245.2,7,73.0,66.0,🔥 Major Value
11,TE2,Sam LaPorta,TE,DET,195.3,195.3,11,60.0,49.0,🔥 Major Value
7,RB3,Breece Hall,RB,NYJ,265.7,265.7,6,27.0,21.0,🔥 Major Value
14,WR5,Malik Nabers,WR,NYG,232.8,232.8,9,29.0,20.0,🔥 Major Value
2,QB3,Jalen Hurts,QB,PHI,355.5,0.0,15,29.0,14.0,🔥 Major Value
12,TE3,Trey McBride,TE,ARZ,188.6,188.6,12,22.0,10.0,🔥 Major Value
8,RB4,Kenneth Walker III,RB,SEA,242.9,242.9,8,17.0,9.0,✅ Slight Value
4,WR2,CeeDee Lamb,WR,DAL,298.4,298.4,2,10.0,8.0,✅ Slight Value



=== 🚫 Top 5 Overpriced Fades ===


,pos_label,player_name,position_proj,team_proj,proj_points,vorp,vorp_rank,search_rank,adp_delta,signal
0,QB1,Josh Allen,QB,BUF,374.2,18.7,13,4.0,-9.0,⚠️ Overpriced
1,QB2,Lamar Jackson,QB,BAL,362.8,7.3,14,11.0,-3.0,⚖️ Fair Value
6,RB1,Bijan Robinson,RB,ATL,280.3,280.3,4,1.0,-3.0,⚖️ Fair Value
3,WR1,Ja'Marr Chase,WR,CIN,312.6,312.6,1,4.0,3.0,⚖️ Fair Value
9,RB2,Saquon Barkley,RB,PHI,271.5,271.5,5,12.0,7.0,✅ Slight Value



=== Signal Distribution ===
signal
🔥 Major Value     8
⚖️ Fair Value     3
✅ Slight Value    3
⚠️ Overpriced     1


## Cell 4 — Output & Validation

Display the full draft board sorted by `vorp_rank` and persist it to
`data/draft_board_latest.csv` for downstream consumption.

In [4]:
# =====================================================================
# Full draft board — sorted by VORP rank
# =====================================================================
output_cols = [
    "vorp_rank", "pos_label", "player_name", "position_proj", "team_proj",
    "proj_points", "baseline_pts", "vorp",
    "search_rank", "adp_delta", "signal",
    "player_id", "full_name", "team_sleeper", "status", "injury_status",
]

draft_board = (
    board.sort_values("vorp_rank")[output_cols]
    .reset_index(drop=True)
)

print("Draft board rows:", len(draft_board))
display(draft_board)

# =====================================================================
# Persist to CSV
# =====================================================================
draft_board.to_csv(DRAFT_BOARD_CSV, index=False)
print("\n\u2705 Saved draft board ->", DRAFT_BOARD_CSV)

# =====================================================================
# Validation report
# =====================================================================
print("\n" + "=" * 60)
print("  VALIDATION REPORT")
print("=" * 60)
print()
print("  Total players on board:", len(draft_board))
print("  Unique player names:", draft_board["player_name"].nunique())
print("  Positions covered:", sorted(draft_board["position_proj"].unique()))

# Position counts
print("\n  --- Position Counts ---")
pos_counts = draft_board["position_proj"].value_counts().sort_index()
for pos, cnt in pos_counts.items():
    print("    " + pos + ":", cnt)

# Starter / flex allocation
print("\n  --- Starter & Flex Allocation ---")
qb_board = pos_counts.get("QB", 0)
rb_board = pos_counts.get("RB", 0)
wr_board = pos_counts.get("WR", 0)
te_board = pos_counts.get("TE", 0)
print("    Dedicated QB slots:    " + str(QB_STARTERS) + "  |  QBs on board:   " + str(qb_board))
print("    Dedicated RB slots:    " + str(RB_STARTERS) + "  |  RBs on board:   " + str(rb_board))
print("    Dedicated WR slots:    " + str(WR_STARTERS) + "  |  WRs on board:   " + str(wr_board))
print("    Dedicated TE slots:    " + str(TE_STARTERS) + "  |  TEs on board:   " + str(te_board))
print("    Flex slots (RB/WR/TE): " + str(FLEX_SPOTS))
print("    Total dedicated:       " + str(DEDICATED_TOTAL))
print("    Total skill starters:  " + str(SKILL_TOTAL))

# VORP stats
print("\n  --- VORP Summary ---")
vmin = draft_board["vorp"].min()
vmax = draft_board["vorp"].max()
vmean = draft_board["vorp"].mean()
vmed = draft_board["vorp"].median()
print("    VORP range:   {:.1f} \u2013 {:.1f}".format(vmin, vmax))
print("    Mean VORP:    {:.2f}".format(vmean))
print("    Median VORP:  {:.2f}".format(vmed))

# Null checks
print("\n  --- Null Checks ---")
for col in ["proj_points", "baseline_pts", "vorp", "search_rank", "adp_delta"]:
    n = draft_board[col].isna().sum()
    status = "\u2705" if n == 0 else "\u26a0\ufe0f {} nulls".format(n)
    print("    {:16s}: {}".format(col, status))

print("\n" + "=" * 60)

Draft board rows: 15


,vorp_rank,pos_label,player_name,position_proj,team_proj,proj_points,baseline_pts,vorp,search_rank,adp_delta,signal,player_id,full_name,team_sleeper,status,injury_status
0,1,WR1,Ja'Marr Chase,WR,CIN,312.6,0.0,312.6,4.0,3.0,⚖️ Fair Value,7564,Ja'Marr Chase,CIN,Active,NaN
1,2,WR2,CeeDee Lamb,WR,DAL,298.4,0.0,298.4,10.0,8.0,✅ Slight Value,6786,CeeDee Lamb,DAL,Active,NaN
2,3,WR3,Tyreek Hill,WR,MIA,287.1,0.0,287.1,145.0,142.0,🔥 Major Value,3321,Tyreek Hill,NaN,Active,Questionable
3,4,RB1,Bijan Robinson,RB,ATL,280.3,0.0,280.3,1.0,-3.0,⚖️ Fair Value,9509,Bijan Robinson,ATL,Active,NaN
4,5,RB2,Saquon Barkley,RB,PHI,271.5,0.0,271.5,12.0,7.0,✅ Slight Value,4866,Saquon Barkley,PHI,Active,NaN
5,6,RB3,Breece Hall,RB,NYJ,265.7,0.0,265.7,27.0,21.0,🔥 Major Value,8155,Breece Hall,NYJ,Active,Questionable
6,7,WR4,Marvin Harrison Jr.,WR,ARZ,245.2,0.0,245.2,73.0,66.0,🔥 Major Value,11628,Marvin Harrison,ARI,Active,NaN
7,8,RB4,Kenneth Walker III,RB,SEA,242.9,0.0,242.9,17.0,9.0,✅ Slight Value,8151,Kenneth Walker,KC,Active,NaN
8,9,WR5,Malik Nabers,WR,NYG,232.8,0.0,232.8,29.0,20.0,🔥 Major Value,11632,Malik Nabers,NYG,Active,Questionable
9,10,TE1,Travis Kelce,TE,KC,210.8,0.0,210.8,92.0,82.0,🔥 Major Value,1466,Travis Kelce,KC,Active,NaN



✅ Saved draft board -> /home/hadev/Projects/Lab/fantasy-footbal-analytics/data/draft_board_latest.csv

  VALIDATION REPORT

  Total players on board: 15
  Unique player names: 15
  Positions covered: ['QB', 'RB', 'TE', 'WR']

  --- Position Counts ---
    QB: 3
    RB: 4
    TE: 3
    WR: 5

  --- Starter & Flex Allocation ---
    Dedicated QB slots:    14  |  QBs on board:   3
    Dedicated RB slots:    28  |  RBs on board:   4
    Dedicated WR slots:    28  |  WRs on board:   5
    Dedicated TE slots:    14  |  TEs on board:   3
    Flex slots (RB/WR/TE): 28
    Total dedicated:       84
    Total skill starters:  112

  --- VORP Summary ---
    VORP range:   0.0 – 312.6
    Mean VORP:    203.81
    Median VORP:  242.90

  --- Null Checks ---
    proj_points     : ✅
    baseline_pts    : ✅
    vorp            : ✅
    search_rank     : ✅
    adp_delta       : ✅

